
# 03 — Baseline Model

## 1. Train / Validation / Test Split

We split the cleaned dataset into three subsets:

- Training: 64%
- Validation: 16%
- Testing: 20%

We use stratified splitting to preserve the class
distribution across all three subsets.

The test set will remain untouched until final evaluation.


In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/processed/cleaned_complaints.csv")

TEXT_COL = "Consumer complaint narrative"
TARGET_COL = "Product"

X = df[TEXT_COL]
y = df[TARGET_COL]

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.20,
    stratify=y_train_val,
    random_state=42
)

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))
print("Total:", len(X_train) + len(X_val) + len(X_test))



## 2. Build the Baseline Pipeline

Our first classification model combines:

1. TF-IDF: Convert complaint text into numerical features.
2. Logistic Regression: Learn patterns and predict product categories.

We use a Scikit-learn Pipeline to keep feature extraction
and classification together and prevent data leakage.


In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        min_df=2,
        ngram_range=(1, 2)
    )),
    ("classifier", LogisticRegression(
        max_iter=1000
    ))
])

baseline_model


## 3. Train the Baseline Model

Train the TF-IDF + Logistic Regression pipeline
using only the training data.

In [ ]:

baseline_model.fit(X_train, y_train)

print("Baseline model trained successfully!")



## 4. Baseline Validation

Evaluate the trained model on the validation set using:

- Accuracy
- Macro F1-score

The test set remains untouched.


In [ ]:

from sklearn.metrics import accuracy_score, f1_score

y_val_pred = baseline_model.predict(X_val)

accuracy = accuracy_score(y_val, y_val_pred)

macro_f1 = f1_score(
    y_val,
    y_val_pred,
    average="macro"
)

print(f"Validation Accuracy: {accuracy:.4f}")
print(f"Validation Macro F1: {macro_f1:.4f}")



## 5. Classification Report

Analyze precision, recall, F1-score, and support
for each product category.

This helps us identify weak classes and understand
how class imbalance affects model performance.


In [ ]:

from sklearn.metrics import classification_report

print(classification_report(
    y_val,
    y_val_pred,
    zero_division=0
))



## 6. Confusion Matrix Analysis

Analyze which product categories the baseline model
confuses with one another.

We use validation predictions only.


In [ ]:

from sklearn.metrics import confusion_matrix

labels = baseline_model.classes_

cm = confusion_matrix(
    y_val,
    y_val_pred,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

cm_df



## 7. Experiment: Balanced Class Weights

Our baseline model performs poorly on some minority classes.

We will train a second model using class_weight="balanced"
and compare its validation performance with the original baseline.

The original baseline model remains unchanged.


In [ ]:

balanced_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        min_df=2,
        ngram_range=(1, 2)
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

balanced_model.fit(X_train, y_train)

print("Balanced model trained successfully!")



## 8. Baseline vs Balanced Model Comparison

Compare both models on the same validation set using:

- Accuracy
- Macro F1-score

The test set remains untouched.


In [ ]:

balanced_val_pred = balanced_model.predict(X_val)

balanced_accuracy = accuracy_score(
    y_val,
    balanced_val_pred
)

balanced_macro_f1 = f1_score(
    y_val,
    balanced_val_pred,
    average="macro"
)

comparison = pd.DataFrame({
    "Model": ["Baseline", "Balanced"],
    "Accuracy": [accuracy, balanced_accuracy],
    "Macro F1": [macro_f1, balanced_macro_f1]
})

comparison.round(4)



## 9. Balanced Model Classification Report

Analyze precision, recall, F1-score, and support
for every product category after applying balanced class weights.

Compare minority-class performance against the baseline.


In [ ]:

print(classification_report(
    y_val,
    balanced_val_pred,
    zero_division=0
))
